# Reference Experiment RT2

This Kaggle notebook is configured for RT2 Mode B only: `EVALUATE_BENCHMARK` evaluates the frozen AI-curated pseudo-GT benchmark. Candidate preparation is intentionally disabled. RT2 is not a production milestone or an official competition evaluation.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
RT2_MODE = os.environ.get("AIC_RT2_MODE", "EVALUATE_BENCHMARK").upper()
STAGE1_INPUT = os.environ.get("AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle")
DATASET_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE1B_INPUT = os.environ.get("AIC_STAGE1B_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports")
STAGE1E_INPUT = os.environ.get("AIC_STAGE1E_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze")
CLIP_INPUT = os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
OPUS_INPUT = os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
BENCHMARK_INPUT = os.environ.get("AIC_RT2_BENCHMARK_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle")
EVALUATION_OUTPUT = Path("/kaggle/working/triage_eg_rt2_evaluation")
print({"mode": RT2_MODE, "stage1": STAGE1_INPUT, "dataset": str(DATASET_ROOT), "benchmark_input": BENCHMARK_INPUT})

In [ ]:
def find_marker_roots(root, marker, max_depth=3):
    root = Path(root)
    matches, frontier = [], [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current)
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend((child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir())
    return matches


def resolve_root(root, marker, max_depth=3):
    matches = find_marker_roots(root, marker, max_depth=max_depth)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one root containing {marker} below {root}; found {matches}")
    return matches[0]


def resolve_input_file(requested_root, filename, search_root=Path("/kaggle/input"), max_depth=6):
    requested = Path(requested_root)
    if requested.is_file():
        if requested.name != filename:
            raise RuntimeError(f"Expected {filename}, got file {requested}")
        return requested
    matches = find_marker_roots(requested, filename, max_depth=max_depth)
    if not matches:
        matches = find_marker_roots(search_root, filename, max_depth=max_depth)
    paths = sorted({root / filename for root in matches})
    if len(paths) != 1:
        raise RuntimeError(
            f"Expected one {filename} below {requested} or {search_root}; found {paths}"
        )
    return paths[0]


if RT2_MODE != "EVALUATE_BENCHMARK":
    raise ValueError("Notebook 12 is Mode B only; set AIC_RT2_MODE=EVALUATE_BENCHMARK")
STAGE1_ROOT = resolve_root(STAGE1_INPUT, "stage1_summary.json")
if not DATASET_ROOT.is_dir():
    raise RuntimeError(f"Missing dataset root: {DATASET_ROOT}")
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "stage1b_summary.json")
STAGE1E_ROOT = resolve_root(STAGE1E_INPUT, "language_path_contract.json")
CLIP_ROOT = resolve_root(CLIP_INPUT, "manifests/asset_manifest.json")
OPUS_ROOT = resolve_root(OPUS_INPUT, "manifests/asset_manifest.json")
BENCHMARK_PATH = resolve_input_file(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl")
print("resolved Stage 1 root:", STAGE1_ROOT)
print("resolved RT2 benchmark:", BENCHMARK_PATH)

In [ ]:
from triage_eg.experiments.reference_rt2 import load_rt2_settings

SETTINGS = load_rt2_settings(REPO_DIR / "configs/experiments/reference_rt2.yaml")
print({"seed": SETTINGS.seed, "lambda_grid": SETTINGS.lambda_grid})

In [ ]:
assert RT2_MODE == "EVALUATE_BENCHMARK"
print("Mode A PREPARE_CANDIDATES: DISABLED")

In [ ]:
if RT2_MODE == "EVALUATE_BENCHMARK":
    from triage_eg.experiments.reference_rt2 import (
        RT2RunnerConfig, create_rt2_evaluation_bundle, load_rt2_benchmark,
        run_reference_rt2_evaluation,
    )
    from triage_eg.retrieval.stage2 import config_from_yaml

    stage2 = config_from_yaml(
        REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml",
        stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT,
        clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT,
        output_root=EVALUATION_OUTPUT / "_stage2_control",
        stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
        build_git_commit=COMMIT,
    )
    QUERIES = load_rt2_benchmark(BENCHMARK_PATH)
    RESULT = run_reference_rt2_evaluation(
        RT2RunnerConfig(stage2, DATASET_ROOT, BENCHMARK_PATH, EVALUATION_OUTPUT, SETTINGS),
        QUERIES,
    )
    ZIP_PATH = Path("/kaggle/working/triage_eg_rt2_evaluation_bundle.zip")
    create_rt2_evaluation_bundle(EVALUATION_OUTPUT, ZIP_PATH)
    print(json.dumps(RESULT, indent=2))

In [ ]:
from IPython.display import Image, display

if (EVALUATION_OUTPUT / "visuals").is_dir():
    for path in sorted((EVALUATION_OUTPUT / "visuals").glob("*_ab.jpg")):
        print(path.stem)
        display(Image(filename=str(path)))

In [ ]:
print("DOWNLOAD ZIP:", ZIP_PATH, "size_bytes=", ZIP_PATH.stat().st_size)
print("RT2_BENCHMARK_STATUS =", RESULT["calibration_status"])
print("DANTE_QUALITY_DECISION = NOT_EVALUATED")